# Week 12: Mini Project v2 — Sensor Log Analyzer (Start) — PHASE 7: Proving Mastery

*Core Mastery: "I can build a complete data pipeline from reading to visualization"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Design a multi-stage data pipeline architecture
2. Read and parse CSV sensor data using `csv` and `io.StringIO`
3. Validate data schemas programmatically
4. Implement data cleaning functions for missing values, invalid entries, and outliers
5. Apply min-max normalization per sensor group
6. Compute summary statistics (mean, std, min, max, count) per sensor
7. Create time-series, histogram, and bar chart visualizations with `matplotlib`
8. Export cleaned data as CSV and summaries as JSON
9. Save plots as PNG image files
10. Orchestrate all stages in a single `main()` function

## 🎯 Core Mastery Connection

This week marks the beginning of your mini project — a complete **Sensor Log Analyzer** pipeline. Over the past 11 weeks you have learned to read files, write functions, handle errors, test code, and visualize data. Now you will combine all of those skills into one cohesive project. The pipeline reads raw sensor CSV data, validates it, cleans it, normalizes it, computes statistics, creates visualizations, and exports everything. By the end of this week you will have a working first version of the full pipeline.

---
## 🧭 Five-Hour Class Roadmap

This notebook is designed for one five-hour class with four short breaks.

| Target | Activity |
|---|---|
| 00:00–00:55 | Concepts and examples → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Concepts and examples → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Concepts and examples → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Concepts and examples → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Core Practice (Exercises 1–8) → Checkpoint 5 |
| 04:45–05:00 | Review and retry failed checks |

Checkpoints provide immediate feedback only inside your Colab runtime. Nothing
is transmitted, saved for grading, or reviewed by the instructor. Exercises 9
and above are optional extensions—not homework.


In [ ]:
# Run this setup cell once at the start of class.
_checkpoint_results = {}

def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = (int(correct), 1)
    if correct:
        print(f"✅ Checkpoint {number}: correct")
        print("Why:", explanation)
    elif not str(answer).strip():
        print(f"🟡 Checkpoint {number}: enter an answer, then run this cell again.")
    else:
        print(f"🔴 Checkpoint {number}: not yet. Review the preceding examples and retry.")
    return correct

def exercise_checkpoint(number, expected=8):
    """Count core exercise cells that contain work and were run in this runtime."""
    import re
    completed = set()
    for source in globals().get("In", []):
        match = re.search(r"#\s*✏️\s*\[EX(\d+)\]", str(source), flags=re.I)
        if not match:
            continue
        answer = re.sub(r"^.*?#\s*✏️\s*\[EX\d+\]", "", str(source), count=1, flags=re.I | re.S).strip()
        if answer and answer != "pass" and "your code here" not in answer.lower():
            completed.add(int(match.group(1)))
    checks = [(index, index in completed) for index in range(1, expected + 1)]
    passed = sum(done for _, done in checks)
    _checkpoint_results[int(number)] = (passed, expected)
    print(f"Core Practice: {passed}/{expected} exercise cells edited and run")
    missing = [str(index) for index, done in checks if not done]
    if not missing:
        print("✅ Core Practice complete.")
    else:
        print("🟡 Still to complete/run:", ", ".join(missing))
    return passed, expected

def show_progress_summary():
    print("\n=== My local progress ===")
    for number in range(1, 6):
        if number in _checkpoint_results:
            passed, total = _checkpoint_results[number]
            print(f"Checkpoint {number}: {passed}/{total}")
        else:
            print(f"Checkpoint {number}: not run")
    print("Results exist only in this temporary runtime.")

print("✅ Local self-check tools ready")


---
## Part 1: Project Overview

The Sensor Log Analyzer processes raw sensor readings through a seven-stage pipeline:

| Stage | Function | Input | Output |
|-------|----------|-------|--------|
| 1. Read | `read_sensor_data()` | CSV string / file | list of dicts |
| 2. Validate | `validate_schema()` | list of dicts | validated list |
| 3. Clean | `clean_missing()`, `remove_invalid()`, `filter_outliers()` | validated list | clean list |
| 4. Normalize | `normalize_per_sensor()` | clean list | normalized list |
| 5. Summarize | `compute_summary()` | normalized list | summary dict |
| 6. Visualize | plotting functions | data + summary | matplotlib figures |
| 7. Export | `export_clean_csv()`, `export_summary_json()` | clean data, summary | files on disk |

**Deliverables:**

| Deliverable | Format | Description |
|-------------|--------|-------------|
| Cleaned CSV | `.csv` | Sensor data after cleaning and normalization |
| Summary JSON | `.json` | Per-sensor statistics |
| Time-series plot | `.png` | Sensor values over time |
| Histogram | `.png` | Distribution of all sensor values |
| Bar chart | `.png` | Mean value per sensor |
| Pipeline script | `.py` / notebook | All code orchestrated in `main()` |

**Figure 1.1** — Pipeline stages printed as a flowchart

In [ ]:
stages = ["Read CSV", "Validate Schema", "Clean Data",
          "Normalize", "Summarize", "Visualize", "Export"]
print("Pipeline Stages:")
print(" → ".join(stages))

**Figure 1.2** — Deliverables checklist printer

In [ ]:
deliverables = {
    "cleaned_data.csv": "Cleaned sensor readings",
    "summary.json": "Per-sensor statistics",
    "timeseries.png": "Time-series plot",
    "histogram.png": "Value distribution",
    "barchart.png": "Per-sensor means"
}
print("Deliverables Checklist:")
for fname, desc in deliverables.items():
    print(f"  [ ] {fname:25s} — {desc}")

---
## Part 2: Sample Sensor Data

We create inline CSV data using `io.StringIO` so the notebook is self-contained. The data includes:
- 20+ rows of sensor readings
- Multiple sensor IDs (S01, S02, S03)
- Messy data: missing values, outlier values (9999), invalid status codes

| Column | Type | Description |
|--------|------|-------------|
| timestamp | str | ISO format datetime |
| sensor_id | str | Sensor identifier |
| value | float | Reading value |
| status | str | OK, WARN, ERROR, or invalid |

**Figure 2.1** — Inline CSV data definition

In [ ]:
import io, csv

RAW_CSV = """timestamp,sensor_id,value,status
2025-06-01 08:00,S01,23.5,OK
2025-06-01 08:00,S02,45.1,OK
2025-06-01 08:00,S03,12.8,OK
2025-06-01 09:00,S01,24.1,OK
2025-06-01 09:00,S02,,WARN
2025-06-01 09:00,S03,13.2,OK
2025-06-01 10:00,S01,9999,ERROR
2025-06-01 10:00,S02,47.3,OK
2025-06-01 10:00,S03,11.9,OK
2025-06-01 11:00,S01,25.0,OK
2025-06-01 11:00,S02,44.8,OK
2025-06-01 11:00,S03,,
2025-06-01 12:00,S01,23.8,OK
2025-06-01 12:00,S02,46.5,WARN
2025-06-01 12:00,S03,14.1,OK
2025-06-01 13:00,S01,24.9,OK
2025-06-01 13:00,S02,9999,OK
2025-06-01 13:00,S03,13.7,OK
2025-06-01 14:00,S01,22.1,INVALID
2025-06-01 14:00,S02,43.2,OK
2025-06-01 14:00,S03,15.0,OK
2025-06-01 15:00,S01,26.3,OK
2025-06-01 15:00,S02,48.0,OK
2025-06-01 15:00,S03,12.5,OK
"""
print("Raw CSV loaded:", len(RAW_CSV.strip().splitlines()), "lines (including header)")

**Figure 2.2** — Parsing CSV into list of dicts

In [ ]:
def parse_csv(csv_string):
    """Parse a CSV string into a list of dictionaries."""
    reader = csv.DictReader(io.StringIO(csv_string.strip()))
    return list(reader)

rows = parse_csv(RAW_CSV)
print(f"Parsed {len(rows)} rows")
for r in rows[:5]:
    print(r)

**Figure 2.3** — Quick data inspection

In [ ]:
# Count issues in the raw data
missing_count = sum(1 for r in rows if r["value"] == "" or r["status"] == "")
outlier_count = sum(1 for r in rows if r["value"] == "9999")
invalid_count = sum(1 for r in rows if r["status"] not in ("OK", "WARN", "ERROR", ""))
print(f"Missing values: {missing_count}")
print(f"Outliers (9999): {outlier_count}")
print(f"Invalid status:  {invalid_count}")

---
### ⏱️ Checkpoint 1 of 5 — Architecture (target 00:55)

Which stage should occur first: read or summarize?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_1_answer = ""  # enter your answer
check_answer(
    1, checkpoint_1_answer, 'read',
    'The pipeline must obtain data before summarizing it.',
)


---
## Part 3: Reading & Validation

Two functions handle the first two pipeline stages:
- `read_sensor_data(csv_string)` — parses CSV, converts value to float
- `validate_schema(records)` — checks required columns exist, flags issues

| Validation Rule | Action |
|----------------|--------|
| Missing column | Raise `ValueError` |
| Non-numeric value | Set to `None` |
| Empty status | Set to `"UNKNOWN"` |

**Figure 3.1** — `read_sensor_data()` with type conversion

In [ ]:
def read_sensor_data(csv_string):
    """Read CSV string, convert value to float or None."""
    reader = csv.DictReader(io.StringIO(csv_string.strip()))
    records = []
    for row in reader:
        try:
            row["value"] = float(row["value"]) if row["value"].strip() else None
        except ValueError:
            row["value"] = None
        records.append(row)
    return records

data = read_sensor_data(RAW_CSV)
print(f"Read {len(data)} records")
print("First record:", data[0])
print("Record with None:", [r for r in data if r["value"] is None][0])

**Figure 3.2** — `validate_schema()` checking required columns

In [ ]:
REQUIRED_COLS = {"timestamp", "sensor_id", "value", "status"}

def validate_schema(records):
    """Validate that all required columns exist and fix empty status."""
    if not records:
        raise ValueError("No records to validate")
    keys = set(records[0].keys())
    missing = REQUIRED_COLS - keys
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    for r in records:
        if not r["status"] or r["status"].strip() == "":
            r["status"] = "UNKNOWN"
    return records

validated = validate_schema(data)
unknown_count = sum(1 for r in validated if r["status"] == "UNKNOWN")
print(f"Validated {len(validated)} records, {unknown_count} set to UNKNOWN")

**Figure 3.3** — Testing validation with bad data

In [ ]:
# Test with missing column
bad_csv = "timestamp,sensor_id,value\n2025-01-01,S01,10\n"
try:
    bad_data = read_sensor_data(bad_csv)
    validate_schema(bad_data)
except ValueError as e:
    print(f"Caught expected error: {e}")

# Test with empty data
try:
    validate_schema([])
except ValueError as e:
    print(f"Caught expected error: {e}")

print("Validation tests passed!")

---
## Part 4: Cleaning Stage

Three cleaning functions remove problematic data:

| Function | Purpose | Method |
|----------|---------|--------|
| `clean_missing()` | Remove rows with `None` values | Filter |
| `remove_invalid()` | Remove rows with invalid status | Filter |
| `filter_outliers()` | Remove statistical outliers | IQR method |

**IQR Method:** Q1 = 25th percentile, Q3 = 75th percentile, IQR = Q3 - Q1. Outliers are values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR.

**Figure 4.1** — `clean_missing()` and `remove_invalid()`

In [ ]:
VALID_STATUSES = {"OK", "WARN", "ERROR"}

def clean_missing(records):
    """Remove records where value is None."""
    cleaned = [r for r in records if r["value"] is not None]
    removed = len(records) - len(cleaned)
    print(f"  clean_missing: removed {removed} rows")
    return cleaned

def remove_invalid(records):
    """Remove records with invalid status codes."""
    cleaned = [r for r in records if r["status"] in VALID_STATUSES]
    removed = len(records) - len(cleaned)
    print(f"  remove_invalid: removed {removed} rows")
    return cleaned

step1 = clean_missing(validated)
step2_data = remove_invalid(step1)
print(f"After cleaning: {len(step2_data)} records remain")

**Figure 4.2** — `filter_outliers()` using IQR

In [ ]:
def filter_outliers(records):
    """Remove outliers using the IQR method across all values."""
    values = sorted(r["value"] for r in records)
    n = len(values)
    q1 = values[n // 4]
    q3 = values[3 * n // 4]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    cleaned = [r for r in records if lower <= r["value"] <= upper]
    removed = len(records) - len(cleaned)
    print(f"  filter_outliers: removed {removed} rows (bounds: {lower:.1f} to {upper:.1f})")
    return cleaned

clean_data = filter_outliers(step2_data)
print(f"Final clean data: {len(clean_data)} records")

**Figure 4.3** — Full cleaning pipeline test

In [ ]:
def clean_pipeline(records):
    """Run all three cleaning stages in sequence."""
    print("Cleaning pipeline:")
    result = clean_missing(records)
    result = remove_invalid(result)
    result = filter_outliers(result)
    print(f"  Pipeline complete: {len(records)} → {len(result)} records")
    return result

final_clean = clean_pipeline(validated)
for r in final_clean[:3]:
    print(f"  {r['timestamp']} | {r['sensor_id']} | {r['value']:6.1f} | {r['status']}")

---
### ⏱️ Checkpoint 2 of 5 — Validation (target 01:55)

Should schema validation precede cleaning? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_2_answer = ""  # enter your answer
check_answer(
    2, checkpoint_2_answer, 'yes',
    'Cleaning relies on known fields and types.',
)


---
## Part 5: Normalization

Min-max normalization scales each sensor's values to the [0, 1] range:

$$x_{norm} = \\frac{x - x_{min}}{x_{max} - x_{min}}$$

We normalize **per sensor_id** so each sensor's range is independent.

**Figure 5.1** — `normalize_per_sensor()` function

In [ ]:
def normalize_per_sensor(records):
    """Apply min-max normalization per sensor_id."""
    # Group by sensor
    groups = {}
    for r in records:
        sid = r["sensor_id"]
        if sid not in groups:
            groups[sid] = []
        groups[sid].append(r)

    normalized = []
    for sid, group in groups.items():
        values = [r["value"] for r in group]
        vmin, vmax = min(values), max(values)
        rng = vmax - vmin if vmax != vmin else 1.0
        for r in group:
            r_copy = dict(r)
            r_copy["value_raw"] = r["value"]
            r_copy["value"] = (r["value"] - vmin) / rng
            normalized.append(r_copy)
        print(f"  Sensor {sid}: range [{vmin:.1f}, {vmax:.1f}] → [0.0, 1.0]")
    return normalized

norm_data = normalize_per_sensor(final_clean)
print(f"\nNormalized {len(norm_data)} records")

**Figure 5.2** — Verifying normalization bounds

In [ ]:
# Check that all normalized values are in [0, 1]
all_values = [r["value"] for r in norm_data]
print(f"Min normalized: {min(all_values):.4f}")
print(f"Max normalized: {max(all_values):.4f}")
assert all(0.0 <= v <= 1.0 for v in all_values), "Normalization failed!"
print("All values in [0, 1] — normalization verified!")

**Figure 5.3** — Displaying normalized vs raw values

In [ ]:
print(f"{'Timestamp':>20} {'ID':>4} {'Raw':>8} {'Norm':>8}")
print("-" * 44)
for r in norm_data[:8]:
    print(f"{r['timestamp']:>20} {r['sensor_id']:>4} {r['value_raw']:8.1f} {r['value']:8.4f}")

---
## Part 6: Summary Statistics

`compute_summary()` returns a dictionary keyed by `sensor_id`, each containing:

| Statistic | Description |
|-----------|-------------|
| `mean` | Average value |
| `std` | Standard deviation |
| `min` | Minimum value |
| `max` | Maximum value |
| `count` | Number of readings |

**Figure 6.1** — `compute_summary()` function

In [ ]:
import math

def compute_summary(records, use_raw=True):
    """Compute per-sensor summary statistics."""
    groups = {}
    for r in records:
        sid = r["sensor_id"]
        val = r.get("value_raw", r["value"]) if use_raw else r["value"]
        if sid not in groups:
            groups[sid] = []
        groups[sid].append(val)

    summary = {}
    for sid, vals in groups.items():
        n = len(vals)
        mean = sum(vals) / n
        variance = sum((v - mean) ** 2 for v in vals) / n
        std = math.sqrt(variance)
        summary[sid] = {
            "mean": round(mean, 2),
            "std": round(std, 2),
            "min": round(min(vals), 2),
            "max": round(max(vals), 2),
            "count": n
        }
    return summary

stats = compute_summary(norm_data)
for sid, s in stats.items():
    print(f"Sensor {sid}: mean={s['mean']}, std={s['std']}, min={s['min']}, max={s['max']}, n={s['count']}")

**Figure 6.2** — Summary as formatted table

In [ ]:
def print_summary_table(summary):
    """Print summary as a formatted table."""
    header = f"{'Sensor':>8} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Count':>6}"
    print(header)
    print("-" * len(header))
    for sid in sorted(summary.keys()):
        s = summary[sid]
        print(f"{sid:>8} {s['mean']:8.2f} {s['std']:8.2f} {s['min']:8.2f} {s['max']:8.2f} {s['count']:6d}")

print_summary_table(stats)

**Figure 6.3** — Validating summary correctness

In [ ]:
# Quick validation
for sid, s in stats.items():
    assert s["min"] <= s["mean"] <= s["max"], f"Stats error for {sid}"
    assert s["count"] > 0, f"Empty group for {sid}"
    assert s["std"] >= 0, f"Negative std for {sid}"
print("All summary statistics validated!")

---
### ⏱️ Checkpoint 3 of 5 — Functions (target 02:55)

Are small pure stages easier to test than one large function? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_3_answer = ""  # enter your answer
check_answer(
    3, checkpoint_3_answer, 'yes',
    'Explicit inputs and outputs support isolated tests.',
)


---
## Part 7: Visualization

Three visualizations summarize the sensor data:
1. **Time-series plot** — sensor values over time, one line per sensor
2. **Histogram** — distribution of all sensor values
3. **Bar chart** — mean value per sensor

**Figure 7.1** — Time-series plot

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt

def plot_timeseries(records):
    """Plot sensor values over time."""
    groups = {}
    for r in records:
        sid = r["sensor_id"]
        if sid not in groups:
            groups[sid] = {"x": [], "y": []}
        groups[sid]["x"].append(r["timestamp"])
        groups[sid]["y"].append(r.get("value_raw", r["value"]))

    fig, ax = plt.subplots(figsize=(10, 5))
    for sid in sorted(groups.keys()):
        ax.plot(groups[sid]["y"], marker="o", label=sid)
    ax.set_title("Sensor Values Over Time")
    ax.set_xlabel("Reading Index")
    ax.set_ylabel("Value")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

fig1 = plot_timeseries(norm_data)
print("Time-series plot created")

**Figure 7.2** — Histogram of all values

In [ ]:
def plot_histogram(records):
    """Plot histogram of all sensor values."""
    values = [r.get("value_raw", r["value"]) for r in records]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(values, bins=15, edgecolor="black", alpha=0.7, color="steelblue")
    ax.set_title("Distribution of Sensor Values")
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

fig2 = plot_histogram(norm_data)
print("Histogram created")

**Figure 7.3** — Bar chart of per-sensor means

In [ ]:
def plot_sensor_means(summary):
    """Bar chart of mean value per sensor."""
    sids = sorted(summary.keys())
    means = [summary[s]["mean"] for s in sids]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(sids, means, color=["#e74c3c", "#3498db", "#2ecc71"], edgecolor="black")
    ax.set_title("Mean Value per Sensor")
    ax.set_xlabel("Sensor ID")
    ax.set_ylabel("Mean Value")
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{val:.1f}", ha="center", fontsize=10)
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    return fig

fig3 = plot_sensor_means(stats)
print("Bar chart created")

---
## Part 8: Export

Final stage: save all outputs to disk.

| Function | Output |
|----------|--------|
| `export_clean_csv()` | `cleaned_data.csv` |
| `export_summary_json()` | `summary.json` |
| `fig.savefig()` | PNG plot files |

**Figure 8.1** — Export functions

In [ ]:
import json as json_mod

def export_clean_csv(records, filepath):
    """Export cleaned records to CSV."""
    if not records:
        print("No records to export")
        return
    keys = ["timestamp", "sensor_id", "value", "status"]
    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(records)
    print(f"Exported {len(records)} rows to {filepath}")

def export_summary_json(summary, filepath):
    """Export summary statistics to JSON."""
    with open(filepath, "w") as f:
        json_mod.dump(summary, f, indent=2)
    print(f"Exported summary to {filepath}")

# Demo (using temp paths)
print("Export functions defined")

**Figure 8.2** — Full pipeline `main()` function

In [ ]:
def main(csv_string, output_dir="/tmp/sensor_output"):
    """Run the complete sensor log analyzer pipeline."""
    import os
    os.makedirs(output_dir, exist_ok=True)
    print("=" * 50)
    print("SENSOR LOG ANALYZER PIPELINE")
    print("=" * 50)

    # Stage 1: Read
    print("\n[1/7] Reading data...")
    records = read_sensor_data(csv_string)
    print(f"  Read {len(records)} records")

    # Stage 2: Validate
    print("\n[2/7] Validating schema...")
    records = validate_schema(records)
    print(f"  Validated {len(records)} records")

    # Stage 3: Clean
    print("\n[3/7] Cleaning data...")
    records = clean_pipeline(records)

    # Stage 4: Normalize
    print("\n[4/7] Normalizing per sensor...")
    records = normalize_per_sensor(records)

    # Stage 5: Summarize
    print("\n[5/7] Computing summary...")
    summary = compute_summary(records)
    print_summary_table(summary)

    # Stage 6: Visualize
    print("\n[6/7] Creating plots...")
    fig1 = plot_timeseries(records)
    fig2 = plot_histogram(records)
    fig3 = plot_sensor_means(summary)
    fig1.savefig(os.path.join(output_dir, "timeseries.png"), dpi=100)
    fig2.savefig(os.path.join(output_dir, "histogram.png"), dpi=100)
    fig3.savefig(os.path.join(output_dir, "barchart.png"), dpi=100)
    plt.close("all")
    print("  Saved 3 plots")

    # Stage 7: Export
    print("\n[7/7] Exporting results...")
    export_clean_csv(records, os.path.join(output_dir, "cleaned_data.csv"))
    export_summary_json(summary, os.path.join(output_dir, "summary.json"))

    print("\n" + "=" * 50)
    print("PIPELINE COMPLETE")
    print("=" * 50)
    return records, summary

# Run the pipeline
result_data, result_summary = main(RAW_CSV)
print(f"\nFinal: {len(result_data)} clean records, {len(result_summary)} sensors")

---
### ⏱️ Checkpoint 4 of 5 — Integration (target 03:55)

What test verifies that stages work together?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_4_answer = ""  # enter your answer
check_answer(
    4, checkpoint_4_answer, 'integration test',
    'Integration tests exercise the connected pipeline.',
)


---
## Exercises

Complete each exercise in the code cell below it. Each cell is marked with `# ✏️ [EXn]`.

### Core Practice and Optional Extension

- **Exercises 1–8:** core in-class practice.
- **Exercises 9 and above:** optional extension; these are not homework.
- Run each completed code cell so Checkpoint 5 can count your local progress.


**EX1:** Write a function `count_sensors(records)` that returns the number of unique sensor IDs.

In [ ]:
# ✏️ [EX1]
def count_sensors(records):
    pass


**EX2:** Write `get_sensor_ids(records)` that returns a sorted list of unique sensor IDs.

In [ ]:
# ✏️ [EX2]
def get_sensor_ids(records):
    pass


**EX3:** Write `filter_by_sensor(records, sensor_id)` that returns only records for the given sensor.

In [ ]:
# ✏️ [EX3]
def filter_by_sensor(records, sensor_id):
    pass


**EX4:** Write `count_missing(records)` that returns the number of records where `value` is `None`.

In [ ]:
# ✏️ [EX4]
def count_missing(records):
    pass


**EX5:** Write `fill_missing_mean(records)` that replaces `None` values with the mean of non-None values for the same sensor.

In [ ]:
# ✏️ [EX5]
def fill_missing_mean(records):
    pass


**EX6:** Write `detect_outliers_iqr(values)` that returns a list of outlier values using the IQR method.

In [ ]:
# ✏️ [EX6]
def detect_outliers_iqr(values):
    pass


**EX7:** Write `zscore_normalize(values)` that returns z-score normalized values: `(x - mean) / std`.

In [ ]:
# ✏️ [EX7]
def zscore_normalize(values):
    pass


**EX8:** Write `compute_median(values)` without using any library — sort the list and pick the middle.

In [ ]:
# ✏️ [EX8]
def compute_median(values):
    pass


---
### ⏱️ Checkpoint 5 of 5 — Core Practice (target 04:45)

Run this after Exercises 1–8. It counts only exercise cells that you edited and
ran in this Colab session. It does not inspect correctness or transmit code.


In [ ]:
exercise_checkpoint(5, expected=8)
show_progress_summary()


---
## 🌟 Optional Extension

Exercises 9 and above are optional enrichment. Stop here if the five-hour class has ended.


**EX9:** Write `sensor_value_range(records)` returning a dict: `{sensor_id: (min_val, max_val)}`.

In [ ]:
# ✏️ [EX9]
def sensor_value_range(records):
    pass


**EX10:** Write `format_timestamp(ts_string)` that converts `'2025-06-01 08:00'` to `'June 1, 2025 08:00'`.

In [ ]:
# ✏️ [EX10]
def format_timestamp(ts_string):
    pass


**EX11:** Write `validate_value_range(records, low, high)` that returns records where `value` is within `[low, high]`.

In [ ]:
# ✏️ [EX11]
def validate_value_range(records, low, high):
    pass


**EX12:** Write `create_sensor_report(summary)` that returns a multi-line string report with one line per sensor.

In [ ]:
# ✏️ [EX12]
def create_sensor_report(summary):
    pass


**EX13:** Write `compare_sensors(summary, sid1, sid2)` that prints which sensor has a higher mean and by how much.

In [ ]:
# ✏️ [EX13]
def compare_sensors(summary, sid1, sid2):
    pass


**EX14:** Write `pipeline_status(records, stage_name)` that prints `Stage: {name} — {n} records`.

In [ ]:
# ✏️ [EX14]
def pipeline_status(records, stage_name):
    pass


**EX15:** Write `export_records_to_dict(records)` that converts the list of records into a dict keyed by sensor_id, each containing a list of values.

In [ ]:
# ✏️ [EX15]
def export_records_to_dict(records):
    pass


---
### 🌉 Bridge to Next Week

You now have a working v1 of the Sensor Log Analyzer. Next week we will:
- Add configurable parameters (CONFIG cell)
- Implement moving average and median filters
- Expand the test suite with 15+ assert tests
- Generate a formatted summary report

Make sure your pipeline runs end-to-end before moving on!